In [1]:
import torch
import ultralytics
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
ultralytics.checks()

Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Setup complete  (16 CPUs, 7.7 GB RAM, 88.4/309.4 GB disk)


In [2]:
import torch
assert torch.cuda.is_available(), "CUDA not available. Install CUDA drivers/toolkit and a CUDA-enabled torch build."
DEVICE = 0
print("gpu:", torch.cuda.get_device_name(0))
print("device:", DEVICE)

gpu: NVIDIA GeForce RTX 3050 6GB Laptop GPU
device: 0


In [3]:
from pathlib import Path

DATA_ROOT = Path(r"d:\\smartnest.ai\\smartnest_media_dataset\\data\\media_dataset\\frames\\RealEstate10K_frames")
ANN_DIR = DATA_ROOT / "annotations"
MANIFEST = ANN_DIR / "manifest.jsonl"
RAW_ROOT = DATA_ROOT.parent.parent / "raw"
HOUSE_ROOM_DIR = RAW_ROOT / "house_room" / "house_room" / "kaggle_room_street_data" / "house_data"
MIT_ROOT = RAW_ROOT / "mit_Dataset" / "mit_Dataset"
YOLO_CLS_DIR = DATA_ROOT / "yolo_cls_v2"
MANIFEST_V2 = ANN_DIR / "manifest_v2.jsonl"
SUMMARY_V2 = ANN_DIR / "summary_v2.json"
LABELS = ["bedroom", "kitchen", "living", "balcony", "bathroom"]

print("manifest:", MANIFEST)
print("raw_root:", RAW_ROOT)
print("house_room_dir:", HOUSE_ROOM_DIR)
print("mit_root:", MIT_ROOT)
print("yolo_cls_dir:", YOLO_CLS_DIR)

manifest: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\annotations\manifest.jsonl
raw_root: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\raw
house_room_dir: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\raw\house_room\house_room\kaggle_room_street_data\house_data
mit_root: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\raw\mit_Dataset\mit_Dataset
yolo_cls_dir: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2


In [4]:
import json
import os
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(42)

def safe_link(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def is_near_blank(path: Path, mean_hi: float = 245.0, mean_lo: float = 10.0, std_max: float = 8.0, flat_std: float = 3.0) -> bool:
    try:
        with Image.open(path) as img:
            img = img.convert("L")
            arr = np.asarray(img)
    except Exception:
        return True
    mean = float(arr.mean())
    std = float(arr.std())
    if std < flat_std:
        return True
    if mean > mean_hi and std < std_max:
        return True
    if mean < mean_lo and std < std_max:
        return True
    return False

def collect_realestate(manifest_path: Path) -> list[dict]:
    records = []
    if not manifest_path.exists():
        return records
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            label = record.get("label")
            if label not in LABELS:
                continue
            image_path = Path(record["image_path"])
            if not image_path.exists():
                continue
            records.append({
                "image_path": str(image_path),
                "label": label,
                "source": "realestate10k"
            })
    return records

def collect_house_room(root_dir: Path) -> list[dict]:
    records = []
    if not root_dir.exists():
        return records
    label_map = {
        "bath": "bathroom",
        "bed": "bedroom",
        "kitchen": "kitchen",
        "living": "living",
    }
    for img_path in root_dir.rglob("*"):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        prefix = img_path.stem.split("_", 1)[0].lower()
        if prefix not in label_map:
            continue
        records.append({
            "image_path": str(img_path),
            "label": label_map[prefix],
            "source": "house_room"
        })
    return records

def collect_mit(root_dir: Path) -> list[dict]:
    records = []
    if not root_dir.exists():
        return records
    mit_classes = {
        "bathroom": "bathroom",
        "bedroom": "bedroom",
        "kitchen": "kitchen",
        "livingroom": "living",
    }
    for split in ["train", "test"]:
        split_root = root_dir / split / split
        for cls_name, mapped in mit_classes.items():
            class_dir = split_root / cls_name
            if not class_dir.exists():
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                records.append({
                    "image_path": str(img_path),
                    "label": mapped,
                    "source": f"mit_{split}"
                })
    return records

records = []
records.extend(collect_realestate(MANIFEST))
records.extend(collect_house_room(HOUSE_ROOM_DIR))
records.extend(collect_mit(MIT_ROOT))

seen = set()
unique_records = []
for rec in records:
    if rec["image_path"] in seen:
        continue
    seen.add(rec["image_path"])
    unique_records.append(rec)

source_counts = Counter((rec["source"], rec["label"]) for rec in unique_records)
print("sources:")
for key, value in source_counts.most_common(12):
    print(key, value)

cleaned = []
blank_skipped = 0
for rec in unique_records:
    if is_near_blank(Path(rec["image_path"])):
        blank_skipped += 1
        continue
    cleaned.append(rec)

by_label = defaultdict(list)
for rec in cleaned:
    by_label[rec["label"]].append(rec)

def split_counts(total: int) -> tuple[int, int, int]:
    if total >= 200:
        min_each = 20
    elif total >= 100:
        min_each = 10
    elif total >= 50:
        min_each = 5
    else:
        min_each = 2 if total >= 10 else 1 if total >= 5 else 0
    val_n = max(int(total * 0.1), min_each)
    test_n = max(int(total * 0.1), min_each)
    if val_n + test_n >= total:
        val_n = max(1, total // 5) if total >= 3 else 0
        test_n = max(1, total // 5) if total - val_n >= 2 else 0
    if val_n + test_n >= total:
        val_n = 1 if total >= 3 else 0
        test_n = 1 if total - val_n >= 2 else 0
    train_n = total - val_n - test_n
    if train_n <= 0:
        train_n = 1
        if val_n > 1:
            val_n -= 1
        elif test_n > 1:
            test_n -= 1
    return train_n, val_n, test_n

final_records = []
split_counts_by_label = {}
for label, items in by_label.items():
    random.shuffle(items)
    train_n, val_n, test_n = split_counts(len(items))
    split_counts_by_label[label] = {"train": train_n, "val": val_n, "test": test_n}
    for rec in items[:train_n]:
        rec["split"] = "train"
        final_records.append(rec)
    for rec in items[train_n:train_n + val_n]:
        rec["split"] = "val"
        final_records.append(rec)
    for rec in items[train_n + val_n:train_n + val_n + test_n]:
        rec["split"] = "test"
        final_records.append(rec)

if YOLO_CLS_DIR.exists():
    shutil.rmtree(YOLO_CLS_DIR)

counts = Counter()
for rec in final_records:
    image_path = Path(rec["image_path"])
    out_path = YOLO_CLS_DIR / rec["split"] / rec["label"] / image_path.name
    safe_link(image_path, out_path)
    counts[(rec["split"], rec["label"])] += 1

with MANIFEST_V2.open("w", encoding="utf-8") as f:
    for rec in final_records:
        json.dump(rec, f)
        f.write("\n")

summary = {
    "total": len(final_records),
    "blank_skipped": blank_skipped,
    "per_label": split_counts_by_label,
    "counts": {
        split: {
            label: counts[(split, label)]
            for label in LABELS
            if (split, label) in counts
        }
        for split in ["train", "val", "test"]
    },
}
with SUMMARY_V2.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("total_linked:", len(final_records))
print("blank_skipped:", blank_skipped)
for key, value in counts.most_common(12):
    print(key, value)

sources:
('house_room', 'living') 1273
('house_room', 'bedroom') 1248
('realestate10k', 'bedroom') 1189
('house_room', 'kitchen') 965
('realestate10k', 'kitchen') 663
('house_room', 'bathroom') 605
('realestate10k', 'bathroom') 395
('realestate10k', 'balcony') 368
('realestate10k', 'living') 201
('mit_train', 'living') 81
('mit_train', 'bathroom') 80
('mit_train', 'bedroom') 80
total_linked: 7300
blank_skipped: 8
('train', 'bedroom') 2030
('train', 'kitchen') 1383
('train', 'living') 1257
('train', 'bathroom') 880
('train', 'balcony') 296
('val', 'bedroom') 253
('test', 'bedroom') 253
('val', 'kitchen') 172
('test', 'kitchen') 172
('val', 'living') 156
('test', 'living') 156
('val', 'bathroom') 110


In [5]:
from ultralytics import YOLO

model = YOLO("yolo26n-cls.pt")
train_results = model.train(
    data=str(YOLO_CLS_DIR),
    epochs=100,
    imgsz=224,
    batch=32,
    device=DEVICE,
    name="room_cls_v2",
    project=str(DATA_ROOT / "runs"),
    workers=4,
    verbose=True,
    patience=15,
    optimizer="AdamW",
    lr0=5e-4,
    lrf=1e-2,
    label_smoothing=0.05,
    cos_lr=True,
    amp=True,
    plots=True,
    seed=42,
    exist_ok=True,
    )
print("train_save_dir:", train_results.save_dir)

New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, 

In [6]:
from ultralytics import YOLO
from pathlib import Path

best_path = Path(train_results.save_dir) / "weights" / "best.pt"
model = YOLO(str(best_path))
metrics = model.val(data=str(YOLO_CLS_DIR), device=DEVICE)
print("val_metrics:", metrics)

Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLO26n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\train... found 5846 images in 5 classes  
val: D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\val... found 727 images in 5 classes  
test: D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\test... found 727 images in 5 classes  
val: Fast image access  (ping: 0.40.1 ms, read: 296.050.4 MB/s, size: 349.8 KB)
val: Scanning D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\val... 727 images, 0 corrupt: 100% ━━━━━━━━━━━━ 727/727  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 46/46 9.7it/s 4.7s<0.1s
                   all   

# Notes

- This notebook uses the auto-labeled frame dataset in `RealEstate10K_frames`.
- The classification dataset is created under `yolo_cls/` with subfolders per class.

In [7]:
from ultralytics import YOLO
from pathlib import Path

best_path = Path(train_results.save_dir) / "weights" / "best.pt"
model = YOLO(str(best_path))
sample = next((YOLO_CLS_DIR / "val").rglob("*.jpg"))
pred = model.predict(source=str(sample), device=DEVICE, save=False, verbose=False)
print("sample:", sample)
print("pred:", pred[0].probs)

sample: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls\val\balcony\0032c663bb6d838e_000000.jpg
pred: ultralytics.engine.results.Probs object with attributes:

data: tensor([2.5600e-03, 2.1242e-02, 8.9960e-01, 7.6546e-02, 5.3704e-05], device='cuda:0')
orig_shape: None
shape: torch.Size([5])
top1: 2
top1conf: tensor(0.8996, device='cuda:0')
top5: [2, 3, 1, 0, 4]
top5conf: tensor([8.9960e-01, 7.6546e-02, 2.1242e-02, 2.5600e-03, 5.3704e-05], device='cuda:0')


In [7]:
from ultralytics import YOLO
from pathlib import Path

best_path = Path(train_results.save_dir) / "weights" / "best.pt"
model = YOLO(str(best_path))
export_path = model.export(format="onnx", imgsz=224, dynamic=True)
print("onnx:", export_path)

Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CPU (13th Gen Intel Core i5-13450HX)
YOLO26n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs

PyTorch: starting from 'D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\runs\room_cls_v2\weights\best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 5) (3.0 MB)

ONNX: starting export with onnx 1.21.0 opset 19...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  20.0s, saved as 'D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\runs\room_cls_v2\weights\best.onnx' (5.9 MB)

Export complete (21.2s)
Results saved to D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\runs\room_cls_v2\weights\best.onnx
Predict:         yolo predict task=classify model=D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\runs\room_cls_v2\weights\best.onnx imgsz

In [3]:
from ultralytics import YOLO
from pathlib import Path

data_root = Path(r"d:\\smartnest.ai\\smartnest_media_dataset\\data\\media_dataset\\frames\\RealEstate10K_frames")
yolo_cls_dir = data_root / "yolo_cls_v2"
default_onnx = data_root / "runs" / "room_cls_v2" / "weights" / "best.onnx"

try:
    onnx_path = Path(train_results.save_dir) / "weights" / "best.onnx"
except Exception:
    onnx_path = default_onnx
if not onnx_path.exists():
    onnx_path = default_onnx

try:
    device = DEVICE
except Exception:
    device = 0

print("onnx_path:", onnx_path)
model = YOLO(str(onnx_path), task="classify")
onnx_metrics = model.val(data=str(yolo_cls_dir), device=device, imgsz=224, task="classify")
print("onnx_val_metrics:", onnx_metrics)

onnx_path: d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\runs\room_cls_v2\weights\best.onnx
Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Loading d:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\runs\room_cls_v2\weights\best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CUDAExecutionProvider
train: D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\train... found 5846 images in 5 classes  
val: D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\val... found 727 images in 5 classes  
test: D:\smartnest.ai\smartnest_media_dataset\data\media_dataset\frames\RealEstate10K_frames\yolo_cls_v2\test... found 727 images in 5 classes  
val: Fast image access  (ping: 0.30.1 ms, read: 48.415.8 MB/s, size: 349.8 KB)
val: Scanning D:\smartnest.ai

# Quality models (cleanliness + cracks)

- Cleanliness model uses the `clean/images` dataset (`clean` vs `messy`).
- Crack model maps both `clean` and `messy` into `clean` (no crack).

In [1]:
# Prepare datasets for cleanliness and crack classifiers
import os, glob, random, yaml
from pathlib import Path

# Self-contained paths so this notebook can run from a fresh kernel
QUALITY_ROOT = Path(r"d:\\smartnest.ai\\quality_models")
QUALITY_RUNS = QUALITY_ROOT / "runs"
CLEANLINESS_DIR = Path(r"d:\\smartnest.ai\\clean\\images")
CRACK_POS_DIR = Path(r"d:\\smartnest.ai\\cracks\\Positive")
CRACK_NEG_DIR = Path(r"d:\\smartnest.ai\\cracks\\Negative")
QUALITY_ROOT.mkdir(parents=True, exist_ok=True)
QUALITY_RUNS.mkdir(parents=True, exist_ok=True)

# Reuse the notebook's helper if it exists; otherwise define a local fallback.
if "safe_link" not in globals():
    import shutil
    def safe_link(src: Path, dst: Path) -> None:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            return
        try:
            os.link(src, dst)
        except OSError:
            shutil.copy2(src, dst)

# --- Cleanliness dataset (use existing split folders under CLEANLINESS_DIR) ---
clean_ds = Path(QUALITY_ROOT)/"cleanliness"
for split in ("train","val","test"):
    for cls in ("clean","messy"):
        (clean_ds/split/cls).mkdir(parents=True, exist_ok=True)

_src = Path(CLEANLINESS_DIR)
for split in ("train","val","test"):
    src_split = _src/split
    if src_split.exists():
        for cls in ("clean","messy"):
            src_cls = src_split/cls
            if src_cls.exists():
                for f in src_cls.iterdir():
                    dest = clean_ds/split/cls/f.name
                    if not dest.exists():
                        safe_link(Path(f), dest)

clean_yaml = str(clean_ds/"data_cleanliness.yaml")
with open(clean_yaml, "w") as fh:
    yaml.dump({
        "train": str(clean_ds/"train"),
        "val": str(clean_ds/"val"),
        "test": str(clean_ds/"test"),
        "nc": 2,
        "names": ["clean","messy"]
    }, fh)

print("Cleanliness dataset prepared:", clean_yaml)

# --- Crack dataset (split Positive -> 'crack', Negative -> 'clean') ---
crack_ds = Path(QUALITY_ROOT)/"crack"
for split in ("train","val","test"):
    for cls in ("crack","clean"):
        (crack_ds/split/cls).mkdir(parents=True, exist_ok=True)

pos_files = sorted(glob.glob(os.path.join(CRACK_POS_DIR, "*")))
neg_files = sorted(glob.glob(os.path.join(CRACK_NEG_DIR, "*")))

def split_list(lst, ratios=(0.8,0.1,0.1)):
    random.shuffle(lst)
    n = len(lst)
    a = int(n * ratios[0])
    b = a + int(n * ratios[1])
    return lst[:a], lst[a:b], lst[b:]

pos_train, pos_val, pos_test = split_list(pos_files)
neg_train, neg_val, neg_test = split_list(neg_files)

for src_list, split in ((pos_train, "train"), (pos_val, "val"), (pos_test, "test")):
    for f in src_list:
        dest = crack_ds/split/"crack"/Path(f).name
        if not dest.exists():
            safe_link(Path(f), dest)

for src_list, split in ((neg_train, "train"), (neg_val, "val"), (neg_test, "test")):
    for f in src_list:
        dest = crack_ds/split/"clean"/Path(f).name
        if not dest.exists():
            safe_link(Path(f), dest)

crack_yaml = str(crack_ds/"data_crack.yaml")
with open(crack_yaml, "w") as fh:
    yaml.dump({
        "train": str(crack_ds/"train"),
        "val": str(crack_ds/"val"),
        "test": str(crack_ds/"test"),
        "nc": 2,
        "names": ["clean","crack"]
    }, fh)

print("Crack dataset prepared:", crack_yaml)


Cleanliness dataset prepared: d:\smartnest.ai\quality_models\cleanliness\data_cleanliness.yaml
Crack dataset prepared: d:\smartnest.ai\quality_models\crack\data_crack.yaml


In [ ]:
# Train cleanliness and crack classifiers on GPU (safer resource handling)
from ultralytics import YOLO
import os
import gc
import torch

# training hyperparams (tune as needed)
epochs = 25
imgsz = 224
batch_clean = 16
batch_crack = 8

# reduce workers for lower memory/IO pressure
workers = 4

device = globals().get('DEVICE', None)
if device is None:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Using device:', device)

# Cleanliness model
clean_model = YOLO("yolo26n-cls.pt")
print('Starting cleanliness training...')
clean_model.train(data=str(clean_ds), epochs=epochs, imgsz=imgsz, batch=batch_clean, device=device, project=QUALITY_RUNS, name="cleanliness", workers=workers, amp=True)

# free GPU memory before starting next training
try:
    del clean_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

# Crack model (smaller batch to avoid OOM)
crack_model = YOLO("yolo26n-cls.pt")
print('Starting crack training...')
crack_model.train(data=str(crack_ds), epochs=epochs, imgsz=imgsz, batch=batch_crack, device=device, project=QUALITY_RUNS, name="crack", workers=workers, amp=True)


Using device: cuda
Starting cleanliness training...
New https://pypi.org/project/ultralytics/8.4.56 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\smartnest.ai\quality_models\cleanliness, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mix

KeyboardInterrupt: 

In [3]:
# Export trained models to ONNX (run after training completes)
from ultralytics import YOLO
import os

for run_name in ("cleanliness","crack"):
    weights = os.path.join(QUALITY_RUNS, run_name, "weights", "best.pt")
    if os.path.exists(weights):
        print('Exporting', run_name)
        m = YOLO(weights)
        onnx_path = m.export(format='onnx', imgsz=224, dynamic=True)
        print('Exported to', onnx_path)
    else:
        print('Weights not found for', run_name, 'at', weights)


Exporting cleanliness
Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CPU (13th Gen Intel Core i5-13450HX)
YOLO26n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs

PyTorch: starting from 'd:\smartnest.ai\quality_models\runs\cleanliness\weights\best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 2) (3.0 MB)

ONNX: starting export with onnx 1.21.0 opset 19...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  3.9s, saved as 'd:\smartnest.ai\quality_models\runs\cleanliness\weights\best.onnx' (5.9 MB)

Export complete (4.3s)
Results saved to D:\smartnest.ai\quality_models\runs\cleanliness\weights\best.onnx
Predict:         yolo predict task=classify model=d:\smartnest.ai\quality_models\runs\cleanliness\weights\best.onnx imgsz=224 
Validate:        yolo val task=classify model=d:\smartnest.ai\quality_models\runs\cleanliness\weights\best.onnx imgsz=224 data=d:\smartnest.ai\quality_models\cleanliness  
Visualize:       https://n

In [6]:
# Test both quality models on their available held-out split
from ultralytics import YOLO
from pathlib import Path

quality_checks = [
    ("cleanliness", Path(clean_ds), Path(QUALITY_RUNS) / "cleanliness" / "weights" / "best.pt", Path(QUALITY_RUNS) / "cleanliness" / "weights" / "best.onnx"),
    ("crack", Path(crack_ds), Path(QUALITY_RUNS) / "crack" / "weights" / "best.pt", Path(QUALITY_RUNS) / "crack" / "weights" / "best.onnx"),
]

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def pick_split(data_dir: Path) -> str:
    test_dir = data_dir / "test"
    test_images = [p for p in test_dir.rglob("*") if p.suffix.lower() in image_exts]
    if test_images:
        return "test"
    return "val"

for name, data_dir, pt_path, onnx_path in quality_checks:
    split_name = pick_split(data_dir)
    print(f"Testing {name} pt on {split_name} split")
    pt_model = YOLO(str(pt_path))
    pt_metrics = pt_model.val(data=str(data_dir), split=split_name, device=device, imgsz=224)
    print(f"{name} pt metrics:", pt_metrics)

    if onnx_path.exists():
        print(f"Testing {name} onnx on {split_name} split")
        onnx_model = YOLO(str(onnx_path), task="classify")
        onnx_metrics = onnx_model.val(data=str(data_dir), split=split_name, device=device, imgsz=224, task="classify")
        print(f"{name} onnx metrics:", onnx_metrics)
    else:
        print(f"Missing ONNX for {name} at {onnx_path}")


Testing cleanliness pt on val split
Ultralytics 8.4.54  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLO26n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
train: D:\smartnest.ai\quality_models\cleanliness\train... found 192 images in 2 classes  
val: D:\smartnest.ai\quality_models\cleanliness\val... found 20 images in 2 classes  
WARNING test: D:\smartnest.ai\quality_models\cleanliness\test... found 0 images in 0 classes (no images found)
val: Fast image access  (ping: 0.70.4 ms, read: 8.42.9 MB/s, size: 157.2 KB)
val: Scanning D:\smartnest.ai\quality_models\cleanliness\val... 20 images, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 6.2s/it 12.4s<40.6s
                   all          1          1
Speed: 1.7ms preprocess, 27.7ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to D:\smartnest.ai\runs\classify\val-6
cleanliness pt met